# Profex/BGMN `.dia` fitting 図の一括作成

`0_4_Profex_fitting_visualization.ipynb` の `.dia` 読み込み処理を関数化し、指定フォルダ内の `.dia` から fitting 図を一括保存します。鉱物色は `0_2_Visualize_Data_251110.ipynb` と同じ対応表を使います。

In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd


## 設定

- `DIA_FOLDER`: `.dia` を探すフォルダ
- `OUTPUT_DIR`: 図の保存先
- `RECURSIVE`: サブフォルダも検索する場合は `True`
- `INCLUDE_OLD`: `OLD` フォルダを含める場合だけ `True`

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "2_Program":
    PROJECT_ROOT = PROJECT_ROOT.parent

DIA_FOLDER = PROJECT_ROOT / "0_Data" / "1_Rietveld" / "SPring-8" / "Mafic" 
# DIA_FOLDER = PROJECT_ROOT / "0_Data" / "1_Rietveld" / "SPring-8" / "Pelitic" 
OUTPUT_DIR = PROJECT_ROOT / "0_Data" / "3_Figure" / "Profex_fitting_visualization"

RECURSIVE = True
INCLUDE_OLD = False
SAVE_PDF = True
SAVE_PNG = True
POSITION = None  # None の場合は abs(Observed - Calculated) の最大値を自動 offset にする


# 線幅設定。Mineral は鉱物 phase curve の線幅。
LINEWIDTHS = {
    "Observed": 0.5,
    "Calculated": 0.5,
    "Background": 1.0,
    "Difference": 2.0,
    "Mineral": 1.0,
}

# legend を表示する場合は True、非表示にする場合は False。
SHOW_LEGEND = True

# 図のサイズ。通常図と範囲指定図の両方で使う。
FIGSIZE = (18, 8)

# True の場合、figsize を維持したまま title/axis label/legend が収まるように自動調整する。
CONSTRAINED_LAYOUT = True

# None は保存サイズを FIGSIZE のまま維持する。切り抜き保存したい場合は "tight" にする。
SAVE_BBOX_INCHES = None
SAVE_PAD_INCHES = 0.15

DIA_FOLDER, OUTPUT_DIR


(PosixPath('/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/1_Rietveld/SPring-8/Mafic'),
 PosixPath('/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization'))

## mineral と色の対応表

`.dia` 内の phase 名を `0_2` と同じ表示用 mineral 名へ変換し、その mineral 名で色を決めます。

In [3]:
list_minerals = [
    "Actinolite", "Na-Amphibole", "Calcite", "Epidote", "Graphite",
    "Hematite", "Quartz", "Rutile", "Ilmenite", "Muscovite", "Titanite", "Pyrite",
    "Albite", "Apatite", "Chlorite", "Amphibole",
]

list_colors_minerals = [
    "#228B22",  # Actinolite
    "#000080",  # Glaucophane -> Na-Amphibole
    "#D2B48C",  # Calcite
    "#808000",  # Epidote
    "#000000",  # Graphite
    "#B22222",  # Hematite
    "#C0C0C0",  # Quartz
    "#FF4500",  # Rutile
    "#3A3A3A",  # Ilmenite
    "#FFD700",  # Muscovite
    "#800080",  # Titanite
    "#B8860B",  # Pyrite
    "#696969",  # Albite
    "#00CED1",  # Apatite
    "#32CD32",  # Chlorite
    "#228B22",  # Amphibole
]

palette_dict_mineral = dict(zip(list_minerals, list_colors_minerals))
unknown_minerals = set()

PHASE_TO_MINERAL = {
    "SiO2p3221": "Quartz",
    "SiO2p3221-2": "Quartz",
    "MUSCOVITE_2M1": "Muscovite",
    "CHLORITE_IIb-2": "Chlorite",
    "Chlorite2b": "Chlorite",
    "Chlorite2bdis": "Chlorite",
    "FLUORAPATITE": "Apatite",
    "Fluorapatite": "Apatite",
    "TiO2r": "Rutile",
    "Graphite-2H": "Graphite",
    "ACTINOLITE": "Actinolite",
    "Actinolite": "Actinolite",
    "Glaucophane": "Na-Amphibole",
    "Winchite": "Amphibole",
    "Plagioclase_Albite": "Albite",
    "Albite": "Albite",
    "Calcite": "Calcite",
    "Epidote": "Epidote",
    "Hematite": "Hematite",
    "Ilmenite": "Ilmenite",
    "Titanite": "Titanite",
    "Pyrite": "Pyrite",
}


def mineral_name(phase_name):
    """`.dia` の phase 名を表示用 mineral 名へ変換する。"""

    mineral = PHASE_TO_MINERAL.get(phase_name, phase_name)
    if mineral not in palette_dict_mineral:
        unknown_minerals.add(mineral)
    return mineral


def mineral_color(mineral):
    """未定義 mineral は灰色で描画して処理を止めない。"""

    return palette_dict_mineral.get(mineral, "#999999")


## `.dia` 読み込み・作図関数

In [4]:
def find_dia_files(folder, recursive=True, include_old=False):
    """指定フォルダ内の `.dia` を探す。macOS の `._*.dia` は除外する。"""

    folder = Path(folder)
    pattern = "**/*.dia" if recursive else "*.dia"
    dia_paths = []
    for path in folder.glob(pattern):
        if path.name.startswith("._"):
            continue
        if not include_old and any(part.lower() == "old" for part in path.parts):
            continue
        dia_paths.append(path)
    return sorted(dia_paths)


def phase_names_from_dia_header(path):
    """`.dia` 1行目の `STRUC[n]=...` から phase curve 名を取り出す。"""

    header = Path(path).read_text(errors="replace").splitlines()[0]
    return re.findall(r"STRUC\[\d+\]=([^\s]+)", header)


def read_dia_file(path, position=None):
    """`.dia` を DataFrame と phase 情報へ変換する。"""

    path = Path(path)
    phase_names = phase_names_from_dia_header(path)
    df = pd.read_csv(path, sep=r"\s+", skiprows=1, header=None)

    phase_col_count = max(0, df.shape[1] - 4)
    if len(phase_names) != phase_col_count:
        phase_names = phase_names[:phase_col_count]
        while len(phase_names) < phase_col_count:
            phase_names.append(f"Curve_{len(phase_names) + 1}")

    df.columns = ["2 theta", "Observed", "Calculated", "Background", *phase_names]
    raw_difference = df["Observed"] - df["Calculated"]
    difference_offset = raw_difference.abs().max() if position is None else position
    df["difference"] = raw_difference - difference_offset

    df_add_background = df.copy()
    if phase_names:
        df_add_background[phase_names] = df_add_background[phase_names].add(df["Background"], axis=0)

    mineral_names = [mineral_name(phase) for phase in phase_names]
    return df, df_add_background, phase_names, mineral_names, difference_offset


def safe_stem(path):
    """保存ファイル名に使う sample 名を作る。"""

    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(path).stem)


def x_range_label(x_min, x_max):
    """x 範囲を保存ファイル名に入れやすい文字列へ変換する。"""

    return f"x{x_min:g}-{x_max:g}".replace(".", "p")


DEFAULT_LINEWIDTHS = {
    "Observed": 0.5,
    "Calculated": 0.5,
    "Background": 1.0,
    "Difference": 2.0,
    "Mineral": 1.0,
}


def merged_linewidths(linewidths=None):
    """作図用の線幅設定を作る。辞書指定で一部だけ上書きできる。"""

    merged = DEFAULT_LINEWIDTHS.copy()
    if linewidths:
        merged.update(linewidths)
    return merged


def plot_common_dia_curves(
    ax, df, df_add_background, phase_names, mineral_names, linewidths=None, difference_offset=0
):
    """通常図と範囲指定図で共通の線種・色・凡例設定を使って描画する。

    `linewidths` には Observed/Calculated/Background/Difference/Mineral の線幅を辞書で渡せる。
    """

    linewidths = merged_linewidths(linewidths=linewidths)
    x = df["2 theta"]
    ax.plot(x, df["Observed"], color="black", linewidth=linewidths["Observed"], alpha=0.7, label="Observed", zorder=5)
    ax.plot(x, df["Calculated"], color="#D62728", linewidth=linewidths["Calculated"], label="Calculated", zorder=4)
    ax.plot(x, df["Background"], color="#808080", linewidth=linewidths["Background"], label="Background", zorder=2)
    # 残差ゼロの位置（オフセット後）を、Difference より薄い灰色で示す。
    ax.axhline(
        y=-difference_offset,
        color="#D3D3D3",
        linewidth=linewidths["Difference"],
        label="_nolegend_",
        zorder=0.5,
    )
    ax.plot(x, df["difference"], color="#A9A9A9", linewidth=linewidths["Difference"], label="Difference", zorder=1)

    used_labels = {"Observed", "Calculated", "Background", "Difference"}
    for phase, mineral in zip(phase_names, mineral_names):
        label = mineral if mineral not in used_labels else "_nolegend_"
        ax.plot(
            x,
            df_add_background[phase],
            color=mineral_color(mineral),
            linewidth=linewidths["Mineral"],
            alpha=0.9,
            label=label,
            zorder=3,
        )
        used_labels.add(mineral)


def visible_x_mask(df, x_min=None, x_max=None):
    """x 範囲指定から表示対象行の mask を作る。"""

    x = df["2 theta"]
    left = x.min() if x_min is None else x_min
    right = x.max() if x_max is None else x_max
    if left > right:
        left, right = right, left
    mask = x.between(left, right)
    if not mask.any():
        raise ValueError(f"No data points in x range: {left} - {right}")
    return mask, left, right


def y_limits_for_visible_range(df, df_add_background, phase_names, mask, margin_ratio=0.08):
    """表示する x 範囲内の全曲線から y 軸範囲を決める。"""

    curve_names = ["Observed", "Calculated", "Background", "difference"]
    values = []
    for column in curve_names:
        values.extend(df.loc[mask, column].dropna().tolist())
    for phase in phase_names:
        values.extend(df_add_background.loc[mask, phase].dropna().tolist())

    y_min = min(values)
    y_max = max(values)
    y_span = y_max - y_min
    margin = y_span * margin_ratio if y_span > 0 else max(abs(y_max), 1) * margin_ratio
    return y_min - margin, y_max + margin


def style_dia_axis(ax, title, x_min, x_max, y_min, y_max, show_legend=True):
    """通常図と範囲指定図で共通の軸・ラベル・凡例フォーマットを設定する。"""

    ax.set_title(title, fontsize=30, pad=10)
    ax.set_xlabel("Diffraction Angle (°2θ, $\\lambda = 0.69992\ \AA$)", fontsize=30, labelpad=10)
    ax.set_ylabel("Intensity (arb. unit)", fontsize=30, labelpad=10)
    ax.tick_params(which="both", direction="in", bottom=True, left=True, top=True, right=True, labelsize=20)
    if show_legend:
        ax.legend(loc="upper right", fontsize=20, frameon=False, ncol=2)
    elif ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)


def save_dia_figure(
    fig,
    path,
    output_dir,
    suffix=None,
    save_pdf=True,
    save_png=True,
    bbox_inches=None,
    pad_inches=0.15,
):
    """PDF/PNG 保存処理を共通化する。"""

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    file_stem = safe_stem(path) if suffix is None else f"{safe_stem(path)}_{suffix}"
    pdf_path = output_dir / f"{file_stem}.pdf"
    png_path = output_dir / f"{file_stem}.png"
    save_kwargs = {"bbox_inches": bbox_inches, "pad_inches": pad_inches}
    if save_pdf:
        fig.savefig(pdf_path, **save_kwargs)
    if save_png:
        fig.savefig(png_path, dpi=300, **save_kwargs)
    return pdf_path, png_path


def plot_dia_file(
    path,
    output_dir,
    position=None,
    save_pdf=True,
    save_png=True,
    show=False,
    linewidths=None,
    show_legend=True,
    figsize=(13, 8),
    constrained_layout=True,
    save_bbox_inches=None,
    save_pad_inches=0.15,
):
    """1つの `.dia` から全 x 範囲の fitting 図を作成して保存する。"""

    path = Path(path)
    df, df_add_background, phase_names, mineral_names, difference_offset = read_dia_file(path, position=position)
    mask, x_min, x_max = visible_x_mask(df)
    y_min, y_max = y_limits_for_visible_range(df, df_add_background, phase_names, mask)

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=constrained_layout)
    plot_common_dia_curves(
        ax,
        df,
        df_add_background,
        phase_names,
        mineral_names,
        linewidths=linewidths,
        difference_offset=difference_offset,
    )
    style_dia_axis(ax, path.stem, x_min, x_max, y_min, y_max, show_legend=show_legend)
    if not constrained_layout:
        fig.tight_layout(pad=save_pad_inches)

    pdf_path, png_path = save_dia_figure(
        fig,
        path,
        output_dir,
        save_pdf=save_pdf,
        save_png=save_png,
        bbox_inches=save_bbox_inches,
        pad_inches=save_pad_inches,
    )
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {
        "sample": path.stem,
        "path": str(path),
        "phase_names": phase_names,
        "mineral_names": mineral_names,
        "difference_offset": difference_offset,
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "output_pdf": str(pdf_path) if save_pdf else None,
        "output_png": str(png_path) if save_png else None,
    }


def plot_dia_file_xrange(
    path,
    output_dir,
    x_min,
    x_max,
    position=None,
    save_pdf=True,
    save_png=True,
    show=False,
    linewidths=None,
    show_legend=True,
    figsize=(13, 8),
    constrained_layout=True,
    save_bbox_inches=None,
    save_pad_inches=0.15,
):
    """指定 x 範囲だけを表示し、その範囲内のデータで difference offset と y 軸を最適化する。"""

    path = Path(path)
    df, _, phase_names, _, _ = read_dia_file(path, position=0)
    mask, x_min, x_max = visible_x_mask(df, x_min=x_min, x_max=x_max)

    # 範囲指定図では、全体ではなく表示範囲内の残差最大値で difference を下げる。
    raw_difference = df["Observed"] - df["Calculated"]
    difference_offset = raw_difference.loc[mask].abs().max() if position is None else position
    df["difference"] = raw_difference - difference_offset

    df_add_background = df.copy()
    if phase_names:
        df_add_background[phase_names] = df_add_background[phase_names].add(df["Background"], axis=0)
    mineral_names = [mineral_name(phase) for phase in phase_names]
    y_min, y_max = y_limits_for_visible_range(df, df_add_background, phase_names, mask)

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=constrained_layout)
    plot_common_dia_curves(
        ax,
        df,
        df_add_background,
        phase_names,
        mineral_names,
        linewidths=linewidths,
        difference_offset=difference_offset,
    )
    style_dia_axis(ax, f"{path.stem} ({x_min:g}-{x_max:g}°2θ)", x_min, x_max, y_min, y_max, show_legend=show_legend)
    if not constrained_layout:
        fig.tight_layout(pad=save_pad_inches)

    suffix = x_range_label(x_min, x_max)
    pdf_path, png_path = save_dia_figure(
        fig,
        path,
        output_dir,
        suffix=suffix,
        save_pdf=save_pdf,
        save_png=save_png,
        bbox_inches=save_bbox_inches,
        pad_inches=save_pad_inches,
    )
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {
        "sample": path.stem,
        "path": str(path),
        "phase_names": phase_names,
        "mineral_names": mineral_names,
        "difference_offset": difference_offset,
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "output_pdf": str(pdf_path) if save_pdf else None,
        "output_png": str(png_path) if save_png else None,
    }


## 一括実行

In [5]:
dia_paths = find_dia_files(DIA_FOLDER, recursive=RECURSIVE, include_old=INCLUDE_OLD)
if not dia_paths:
    raise ValueError("No .dia files found. DIA_FOLDER を確認してください。")

summary_rows = []
for dia_path in dia_paths:
    summary_rows.append(
        plot_dia_file(
            dia_path,
            OUTPUT_DIR,
            position=POSITION,
            save_pdf=SAVE_PDF,
            save_png=SAVE_PNG,
            show=False,
            linewidths=LINEWIDTHS,
            show_legend=SHOW_LEGEND,
            figsize=FIGSIZE,
            constrained_layout=CONSTRAINED_LAYOUT,
            save_bbox_inches=SAVE_BBOX_INCHES,
            save_pad_inches=SAVE_PAD_INCHES,
        )
    )

summary_df = pd.DataFrame(summary_rows)
print(f"Created figures for {len(summary_df)} .dia file(s).")
print(f"Output: {OUTPUT_DIR.resolve()}")
if unknown_minerals:
    print("Unknown minerals drawn in gray:", sorted(unknown_minerals))

summary_df


Created figures for 26 .dia file(s).
Output: /Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization


,sample,path,phase_names,mineral_names,difference_offset,x_min,x_max,y_min,y_max,output_pdf,output_png
0,SP-MTN-42,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",7310.93,4.5,40.0,-26029.0704,139375.4804,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
1,SP-MTN-43,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",8924.45,3.9,40.0,-34206.9880,202985.2880,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
2,SP-MTN-44,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",2912.75,3.9,40.0,-11326.3112,68435.4512,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
3,SP-MTN-45,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",1301.19,3.9,40.0,-6136.3712,45106.5012,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
4,SP-MTN-46,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",2482.29,3.9,40.0,-7728.0600,46085.2100,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
5,SP-MTN-47,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",6025.24,3.9,40.0,-21797.4368,119533.4368,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
6,SP-MTN-56,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",5034.60,3.9,40.0,-20945.5680,136761.7680,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
7,SP-MTN-57,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",6144.02,3.9,40.0,-23948.2584,145124.9084,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
8,SP-MTN-58,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",5938.05,3.9,40.0,-23952.1480,151150.5480,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
9,SP-MTN-59,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",5699.22,3.9,40.0,-24408.3456,164235.2856,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...


## 出力確認

In [6]:
display(summary_df[["sample", "difference_offset", "phase_names", "mineral_names", "output_pdf", "output_png"]].head())
print("unknown_minerals:", sorted(unknown_minerals))


,sample,difference_offset,phase_names,mineral_names,output_pdf,output_png
0,SP-MTN-42,7310.93,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
1,SP-MTN-43,8924.45,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
2,SP-MTN-44,2912.75,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
3,SP-MTN-45,1301.19,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
4,SP-MTN-46,2482.29,"[SiO2p3221, MUSCOVITE_2M1, Titanite, Fluorapat...","[Quartz, Muscovite, Titanite, Apatite, Epidote...",/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...


unknown_minerals: []


## x 軸範囲指定版

`X_RANGES` に指定した範囲だけを表示し、y 軸はその範囲内に見えている曲線の値から自動で最適化します。基本の線種・色・ラベルは通常図と同じ helper を使います。

In [7]:
RECURSIVE = True
INCLUDE_OLD = False
SAVE_PDF = True
SAVE_PNG = True
POSITION = None  # None の場合は abs(Observed - Calculated) の最大値を自動 offset にする


# 線幅設定。Mineral は鉱物 phase curve の線幅。
LINEWIDTHS = {
    "Observed": 1,
    "Calculated": 1,
    "Background": 1.0,
    "Difference": 2.0,
    "Mineral": 4.0,
}

# legend を表示する場合は True、非表示にする場合は False。
SHOW_LEGEND = False

# 図のサイズ。通常図と範囲指定図の両方で使う。
FIGSIZE = (10, 8)


In [8]:
# 例: 9.5-10.5°2θ と 20-22°2θ を拡大する。必要な範囲に編集してください。
X_RANGES = [
    (4.30, 4.90),
    (9.7, 14.0),
    (11.63, 11.7),
]

RANGE_OUTPUT_DIR = OUTPUT_DIR / "xrange"
X_RANGES, RANGE_OUTPUT_DIR


([(4.3, 4.9), (9.7, 14.0), (11.63, 11.7)],
 PosixPath('/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization/xrange'))

In [9]:
range_summary_rows = []
for dia_path in dia_paths:
    for x_min, x_max in X_RANGES:
        range_summary_rows.append(
            plot_dia_file_xrange(
                dia_path,
                RANGE_OUTPUT_DIR,
                x_min=x_min,
                x_max=x_max,
                position=POSITION,
                save_pdf=SAVE_PDF,
                save_png=SAVE_PNG,
                show=False,
                linewidths=LINEWIDTHS,
                show_legend=SHOW_LEGEND,
                figsize=FIGSIZE,
                constrained_layout=CONSTRAINED_LAYOUT,
                save_bbox_inches=SAVE_BBOX_INCHES,
                save_pad_inches=SAVE_PAD_INCHES,
            )
        )

range_summary_df = pd.DataFrame(range_summary_rows)
print(f"Created ranged figures for {len(range_summary_df)} plot(s).")
print(f"Output: {RANGE_OUTPUT_DIR.resolve()}")
range_summary_df[["sample", "x_min", "x_max", "y_min", "y_max", "output_pdf", "output_png"]].head()


Created ranged figures for 78 plot(s).
Output: /Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization/xrange


,sample,x_min,x_max,y_min,y_max,output_pdf,output_png
0,SP-MTN-42,4.30,4.9,-330.3512,1673.7112,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
1,SP-MTN-42,9.70,14.0,-26029.0704,139375.4804,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
2,SP-MTN-42,11.63,11.7,-402.7040,2734.8640,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
3,SP-MTN-43,4.30,4.9,-377.3348,1992.0348,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
4,SP-MTN-43,9.70,14.0,-34206.9880,202985.2880,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...


## x・y 軸範囲指定版

上の import・設定・mineral 対応表・作図関数のセルを実行してから、この節を実行してください。
`X_RANGES` と同じ順序で `Y_RANGES` に `(下限, 上限)` を指定します。
例えば `(-1000, 10000)` で y 軸を固定できます。各端点を `None` にすると、表示する x 範囲のデータから自動設定します。
`Y_RANGES` の要素数は `X_RANGES` と揃えてください。
線幅・凡例・図サイズなどは上の設定を引き継ぎ、図は `OUTPUT_DIR / "xyrange"` に保存します。

`MINERAL_FILL_ALPHA` に表示用鉱物名と alpha（0〜1）を指定すると、その鉱物の phase curve と Background の間を鉱物色で塗りつぶします。
例: `{"Quartz": 0.3, "Albite": 0.2}`。`{}` で塗りつぶしを無効にできます。
この設定はこの節の XY 範囲指定版のみに適用されます。同名鉱物の複数 phase は個別に塗りつぶします。


In [10]:
def plot_dia_file_xyrange(
    path,
    output_dir,
    x_min,
    x_max,
    y_min=None,
    y_max=None,
    position=None,
    save_pdf=True,
    save_png=True,
    show=False,
    linewidths=None,
    show_legend=True,
    figsize=(13, 8),
    constrained_layout=True,
    save_bbox_inches=None,
    save_pad_inches=0.15,
    mineral_fill_alpha=None,
):
    """指定した x/y 範囲で作図する。y の各端点が None の場合は自動設定する。"""

    # 表示用鉱物名ごとに alpha を指定する。未指定の鉱物は塗りつぶさない。
    fill_alpha = {} if mineral_fill_alpha is None else dict(mineral_fill_alpha)
    for mineral, alpha in fill_alpha.items():
        alpha = float(alpha)
        if not 0 <= alpha <= 1:
            raise ValueError(f"Fill alpha must be between 0 and 1: {mineral}={alpha}")
        fill_alpha[mineral] = alpha

    path = Path(path)
    df, _, phase_names, _, _ = read_dia_file(path, position=0)
    mask, x_min, x_max = visible_x_mask(df, x_min=x_min, x_max=x_max)

    # 範囲指定図では、全体ではなく表示範囲内の残差最大値で difference を下げる。
    raw_difference = df["Observed"] - df["Calculated"]
    difference_offset = raw_difference.loc[mask].abs().max() if position is None else position
    df["difference"] = raw_difference - difference_offset

    df_add_background = df.copy()
    if phase_names:
        df_add_background[phase_names] = df_add_background[phase_names].add(df["Background"], axis=0)
    mineral_names = [mineral_name(phase) for phase in phase_names]
    auto_y_min, auto_y_max = y_limits_for_visible_range(df, df_add_background, phase_names, mask)
    y_min = auto_y_min if y_min is None else float(y_min)
    y_max = auto_y_max if y_max is None else float(y_max)
    from math import isfinite
    if not (isfinite(y_min) and isfinite(y_max) and y_min < y_max):
        raise ValueError(f"Y range must be finite and y_min < y_max: {y_min}, {y_max}")

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=constrained_layout)
    plot_common_dia_curves(
        ax,
        df,
        df_add_background,
        phase_names,
        mineral_names,
        linewidths=linewidths,
        difference_offset=difference_offset,
    )
    # phase curve は background 加算済み。background と各 phase curve の間を塗る。
    # 同じ表示用鉱物名に複数 phase がある場合は、それぞれを塗りつぶす。
    for phase, mineral in zip(phase_names, mineral_names):
        if mineral in fill_alpha:
            ax.fill_between(
                df["2 theta"].to_numpy(),
                df["Background"].to_numpy(),
                df_add_background[phase].to_numpy(),
                color=mineral_color(mineral),
                alpha=fill_alpha[mineral],
                linewidth=0,
                label="_nolegend_",
                zorder=1.5,
            )
    style_dia_axis(ax, f"{path.stem} ({x_min:g}-{x_max:g}°2θ)", x_min, x_max, y_min, y_max, show_legend=show_legend)
    if not constrained_layout:
        fig.tight_layout(pad=save_pad_inches)

    y_label = f"y{y_min:g}-{y_max:g}".replace(".", "p")
    suffix = f"{x_range_label(x_min, x_max)}_{y_label}"
    pdf_path, png_path = save_dia_figure(
        fig,
        path,
        output_dir,
        suffix=suffix,
        save_pdf=save_pdf,
        save_png=save_png,
        bbox_inches=save_bbox_inches,
        pad_inches=save_pad_inches,
    )
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {
        "sample": path.stem,
        "path": str(path),
        "phase_names": phase_names,
        "mineral_names": mineral_names,
        "difference_offset": difference_offset,
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "output_pdf": str(pdf_path) if save_pdf else None,
        "output_png": str(png_path) if save_png else None,
    }


In [11]:
RECURSIVE = True
INCLUDE_OLD = False
SAVE_PDF = True
SAVE_PNG = True
POSITION = None  # None の場合は abs(Observed - Calculated) の最大値を自動 offset にする


# 線幅設定。Mineral は鉱物 phase curve の線幅。
LINEWIDTHS = {
    "Observed": 1,
    "Calculated": 1,
    "Background": 1.0,
    "Difference": 2.0,
    "Mineral": 4.0,
}

# legend を表示する場合は True、非表示にする場合は False。
SHOW_LEGEND = False

# 図のサイズ。通常図と範囲指定図の両方で使う。
FIGSIZE = (9, 8)

In [12]:
X_RANGES = [
    (14.27, 14.52),
]

# X_RANGES は上の x 軸範囲設定セルの値を使用します。
# 各 x 範囲に対応する y 範囲。例: (-1000, 10000)
Y_RANGES = [
    (800, 3000),
]

# XY 範囲指定版のみ: 表示用鉱物名と塗りつぶしの alpha（0〜1）。
# 鉱物の色は上の対応表と共通。{} にすると塗りつぶしなし。
# 指定鉱物を含まない試料では、その鉱物の塗りつぶしはスキップされます。
MINERAL_FILL_ALPHA = {
    "Apatite": 0.3,
    # "Albite": 0.2,
    # "Muscovite": 0.4,
}

POSITION=-1000

XY_RANGE_OUTPUT_DIR = OUTPUT_DIR / "xyrange"
list(zip(X_RANGES, Y_RANGES)), XY_RANGE_OUTPUT_DIR


([((14.27, 14.52), (800, 3000))],
 PosixPath('/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization/xyrange'))

In [13]:
if len(X_RANGES) != len(Y_RANGES):
    raise ValueError("X_RANGES と Y_RANGES の要素数を揃えてください。")

xy_dia_paths = find_dia_files(DIA_FOLDER, recursive=RECURSIVE, include_old=INCLUDE_OLD)
if not xy_dia_paths:
    raise ValueError("No .dia files found. DIA_FOLDER を確認してください。")

xy_range_summary_rows = []
for dia_path in xy_dia_paths:
    for (x_min, x_max), (y_min, y_max) in zip(X_RANGES, Y_RANGES):
        xy_range_summary_rows.append(
            plot_dia_file_xyrange(
                dia_path,
                XY_RANGE_OUTPUT_DIR,
                x_min=x_min,
                x_max=x_max,
                y_min=y_min,
                y_max=y_max,
                position=POSITION,
                save_pdf=SAVE_PDF,
                save_png=SAVE_PNG,
                show=False,
                linewidths=LINEWIDTHS,
                show_legend=SHOW_LEGEND,
                figsize=FIGSIZE,
                constrained_layout=CONSTRAINED_LAYOUT,
                save_bbox_inches=SAVE_BBOX_INCHES,
                save_pad_inches=SAVE_PAD_INCHES,
                mineral_fill_alpha=MINERAL_FILL_ALPHA,
            )
        )

xy_range_summary_df = pd.DataFrame(xy_range_summary_rows)
print(f"Created x/y ranged figures for {len(xy_range_summary_df)} plot(s).")
print(f"Output: {XY_RANGE_OUTPUT_DIR.resolve()}")
xy_range_summary_df.reindex(
    columns=["sample", "x_min", "x_max", "y_min", "y_max", "output_pdf", "output_png"]
).head()


Created x/y ranged figures for 26 plot(s).
Output: /Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization/xyrange


,sample,x_min,x_max,y_min,y_max,output_pdf,output_png
0,SP-MTN-42,14.27,14.52,800.0,3000.0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
1,SP-MTN-43,14.27,14.52,800.0,3000.0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
2,SP-MTN-44,14.27,14.52,800.0,3000.0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
3,SP-MTN-45,14.27,14.52,800.0,3000.0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
4,SP-MTN-46,14.27,14.52,800.0,3000.0,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...


In [14]:
X_RANGES = [
    (4.30, 4.90),
]

# X_RANGES は上の x 軸範囲設定セルの値を使用します。
# 各 x 範囲に対応する y 範囲。例: (-1000, 10000)
Y_RANGES = [
    (None, None),
]

# XY 範囲指定版のみ: 表示用鉱物名と塗りつぶしの alpha（0〜1）。
# 鉱物の色は上の対応表と共通。{} にすると塗りつぶしなし。
# 指定鉱物を含まない試料では、その鉱物の塗りつぶしはスキップされます。
MINERAL_FILL_ALPHA = {
    "Actinolite": 0.3,
    "Na-Amphibole": 0.3,
    "Epidote":0.3,
    # "Albite": 0.2,
    # "Muscovite": 0.4,
}

POSITION=None

XY_RANGE_OUTPUT_DIR = OUTPUT_DIR / "xyrange"
list(zip(X_RANGES, Y_RANGES)), XY_RANGE_OUTPUT_DIR

if len(X_RANGES) != len(Y_RANGES):
    raise ValueError("X_RANGES と Y_RANGES の要素数を揃えてください。")

xy_dia_paths = find_dia_files(DIA_FOLDER, recursive=RECURSIVE, include_old=INCLUDE_OLD)
if not xy_dia_paths:
    raise ValueError("No .dia files found. DIA_FOLDER を確認してください。")

xy_range_summary_rows = []
for dia_path in xy_dia_paths:
    for (x_min, x_max), (y_min, y_max) in zip(X_RANGES, Y_RANGES):
        xy_range_summary_rows.append(
            plot_dia_file_xyrange(
                dia_path,
                XY_RANGE_OUTPUT_DIR,
                x_min=x_min,
                x_max=x_max,
                y_min=y_min,
                y_max=y_max,
                position=POSITION,
                save_pdf=SAVE_PDF,
                save_png=SAVE_PNG,
                show=False,
                linewidths=LINEWIDTHS,
                show_legend=SHOW_LEGEND,
                figsize=FIGSIZE,
                constrained_layout=CONSTRAINED_LAYOUT,
                save_bbox_inches=SAVE_BBOX_INCHES,
                save_pad_inches=SAVE_PAD_INCHES,
                mineral_fill_alpha=MINERAL_FILL_ALPHA,
            )
        )

xy_range_summary_df = pd.DataFrame(xy_range_summary_rows)
print(f"Created x/y ranged figures for {len(xy_range_summary_df)} plot(s).")
print(f"Output: {XY_RANGE_OUTPUT_DIR.resolve()}")
xy_range_summary_df.reindex(
    columns=["sample", "x_min", "x_max", "y_min", "y_max", "output_pdf", "output_png"]
).head()



Created x/y ranged figures for 26 plot(s).
Output: /Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_Data/3_Figure/Profex_fitting_visualization/xyrange


,sample,x_min,x_max,y_min,y_max,output_pdf,output_png
0,SP-MTN-42,4.3,4.9,-330.3512,1673.7112,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
1,SP-MTN-43,4.3,4.9,-377.3348,1992.0348,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
2,SP-MTN-44,4.3,4.9,-4565.1984,24055.7484,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
3,SP-MTN-45,4.3,4.9,-3775.5752,20646.1252,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
4,SP-MTN-46,4.3,4.9,-4090.8496,20888.7296,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...,/Volumes/SSD-PHPU3A/研究/0_DATA/6_EPMA_XRD検証/0_D...
